In [1]:
import pandas as pd
import numpy as np
import unicodedata
from sklearn.model_selection import train_test_split

# 1. Carga del Dataset
df = pd.read_csv('datascience/data/raw/gym_churn_us.csv')

# --- LIMPIEZA Y SANEAMIENTO ---

# Función para normalización de texto
def normalize_string(s):
    if isinstance(s, str):
        s = s.strip().lower() # Trim & Lowercase
        # Normalización Unicode: Eliminar acentos y tildes
        s = "".join(c for c in unicodedata.normalize('NFD', s)
                    if unicodedata.category(c) != 'Mn')
    return s

# Aplicar saneamiento a columnas tipo objeto (si las hubiera)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(normalize_string)

# 2. Integridad de Tipos y Duplicados
# Casteo estricto: Optimizamos tipos de datos numéricos
for col in df.columns:
    if df[col].dtype in ['float64', 'int64']:
        if (df[col] % 1 == 0).all():
            df[col] = df[col].astype('int64') # Enteros puros
        else:
            df[col] = df[col].astype('float64') # Continuos

# Gestión de Duplicados (Filas idénticas)
df = df.drop_duplicates()

# 3. Tratamiento de Valores Faltantes
# Imputación Inteligente
for col in df.columns:
    if df[col].isnull().any():
        if df[col].dtype in ['float64', 'int64']:
            # Mediana para numéricos (mitiga efecto de outliers)
            df[col] = df[col].fillna(df[col].median())
        else:
            # Moda para categóricos
            df[col] = df[col].fillna(df[col].mode()[0])

# --- INGENIERÍA DE COLUMNAS (FEATURE SELECTION) ---

# Eliminación de Ruido (>70% nulos o varianza cero)
cols_to_drop = [col for col in df.columns if (df[col].isnull().mean() > 0.7) or (df[col].nunique() <= 1)]
df.drop(columns=cols_to_drop, inplace=True)

# 4. Tratamiento de Outliers (Método IQR)
# Aplicado a variables continuas clave para no "ensuciar" el promedio
numeric_cont = ['Age', 'Avg_additional_charges_total', 'Lifetime',
                'Avg_class_frequency_total', 'Avg_class_frequency_current_month']

for col in numeric_cont:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    # Capamos valores (clipping) para mantener la integridad de la fila
    df[col] = np.clip(df[col], lower, upper)

# --- ACTIVIDADES ADICIONALES ---

# Cambio a camelCase
def to_camel_case(text):
    parts = text.split('_')
    return parts[0].lower() + ''.join(x.title() for x in parts[1:])

df.columns = [to_camel_case(c) for c in df.columns]

# Partición Estratificada (Random State 42 inmutable)
X = df.drop('churn', axis=1)
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Guardar Dataset Limpio
df.to_csv('gym_churn_cleaned.csv', index=False)